## MHW detection: percentile thresholds

Implementing the Hobday et al. (2016) methodology for marine heatwave detection. Starting with a small preview: the 90th percentile SST for a single calendar day, across all available years, before generalizing to every day of the year.

June 15 is used here rather than a peak-summer day, since it falls before the June 21 cutoff, so it has 11 years of coverage, unlike days after June 21, which have only 10. (see year coverage note in notebook 02).

In [1]:
import duckdb
import pandas as pd

In [2]:
con = duckdb.connect()

In [3]:
# 90th percentile SST for June 15, across all 11 years in the dataset
result = con.execute("""
    SELECT PERCENTILE_CONT(0.9) WITHIN GROUP (ORDER BY analysed_sst) - 273.15 AS p90_celsius,
           COUNT(*) AS n_observations
    FROM '../data/processed/med_sst_2016_2026.parquet'
    WHERE MONTH(time) = 6 AND DAY(time) = 15
""").df()

print(result)

   p90_celsius  n_observations
0    24.469993          131296


**Results**: 24.47 C, calculated from 131,296 observations (11 years x ~11,936 sea cells per day). As expected, this is well above the June average of 22.2 C established in notebook 02: the threshold is meant to mark the boundary between normal and exceptional, not represent a typical day.

In [4]:
# 90th percentile SST across all available years (varies by calendar day, see note below)
result = con.execute("""
    SELECT MONTH(time) AS month, DAY(time) AS day, PERCENTILE_CONT(0.9) WITHIN GROUP (ORDER BY analysed_sst) - 273.15 AS p90_celsius,
           COUNT(*) AS n_observations
    FROM '../data/processed/med_sst_2016_2026.parquet'
    GROUP BY month, day
    ORDER BY month, day
""").df()

print(result)

     month  day  p90_celsius  n_observations
0        1    1    16.239994          131296
1        1    2    16.159994          131296
2        1    3    16.059994          131296
3        1    4    16.009994          131296
4        1    5    15.939994          131296
..     ...  ...          ...             ...
361     12   27    16.509994          119360
362     12   28    16.419994          119360
363     12   29    16.329994          119360
364     12   30    16.339994          119360
365     12   31    16.289994          119360

[366 rows x 4 columns]


In [5]:
# checking the result against the 15th June 90th percentile SST
print(result[(result["month"] == 6) & (result["day"] == 15)])

     month  day  p90_celsius  n_observations
166      6   15    24.469993          131296


**Results**: 24.469993 C, matching exactly the isolated single-day calculation from earlier in the notebook. Confirms the grouped query (366 daily thresholds) produces the same result as filtering for one specific day manually, no discrepancy introduced by the GROUP BY.

In [6]:
# Temporarily raise the row display limit to inspect all 366 days at once
with pd.option_context("display.max_rows", None):
    print(result)

     month  day  p90_celsius  n_observations
0        1    1    16.239994          131296
1        1    2    16.159994          131296
2        1    3    16.059994          131296
3        1    4    16.009994          131296
4        1    5    15.939994          131296
5        1    6    15.849994          131296
6        1    7    15.839994          131296
7        1    8    15.809994          131296
8        1    9    15.709994          131296
9        1   10    15.659994          131296
10       1   11    15.589994          131296
11       1   12    15.529994          131296
12       1   13    15.459994          131296
13       1   14    15.399994          131296
14       1   15    15.319994          131296
15       1   16    15.179994          131296
16       1   17    15.119994          131296
17       1   18    15.149994          131296
18       1   19    15.079994          131296
19       1   20    14.989994          131296
20       1   21    14.989994          131296
21       1

**Results**: 366 daily thresholds calculated, one per calendar day. Verified against the isolated June 15 calculation from earlier in the notebook (24.47 C in both cases), confirming the grouped query produces consistent results. Observation counts vary by exact calendar day, not by month: days from January 1 through June 21 have 131,296 observations (11 years of coverage), while June 22 through December 31 have 119,360 (10 years): the cutoff falls mid-June because the dataset ends 2026-06-21, not at a month boundary.